# CLV 총수준·N/V 구성비 표현 M2 — Dunnhumby seed 42

직전 고정 구성비 구조가 잃었던 historical CLV proxy의 총수준을 복원한 역사적 개발구간 실험입니다.

- 학습: Dunnhumby DAY 1~683
- 평가: DAY 684~690 신규상품
- 표현: ID 64차원 + 거래활동 N 4차원 + 거래당 가치 V 4차원
- 사용자 CLV 총수준: `q_C = Percentile(N×V)`
- 사용자별 N/V 구성비: `pi_N, pi_V = softmax(q_N, q_V)`
- 축별 실제 배분: `b_N=q_C·pi_N`, `b_V=q_C·pi_V`, 따라서 `b_N+b_V=q_C`
- 최대 축 규모: 학습하지 않는 고정값 `rho=0.05`
- 아이템 N/V: 구매자의 N/V 성향에서 인기도·카테고리 기대치를 제거한 train-only 잔차
- 고정: binary graph, uniform negative sampling, plain pairwise BPR, 100 epoch, 하나의 optimizer
- 비교: protocol이 같은 기존 M1@64 결과만 재사용

이 실행은 이미 분리된 역사적 개발구간의 seed 42 탐색입니다. 최종 test와 holdout을 생성하지 않으며 통계적 유의성을 주장하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '7494380ef61b321689dfd8d104fecd897c62fe72'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)


In [ ]:
import json
import torch
from lightgcn_clv_fixed_composition_nv import (
    configure_fixed_composition_run,
    preflight_summary,
    run_fixed_composition_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_fixed_composition_run(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_total_level_composition_nv_historical_screen_v2'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['historical_development_split']['final_test_constructed'] is False
assert summary['historical_development_split']['holdout_constructed'] is False
assert summary['m2']['architecture'] == 'ID(64)|activity(4)|transaction-value(4)'
assert summary['m2']['fixed_max_axis_scale'] == 0.05
assert 'q_C' in summary['m2']['user_total_axis_level']
assert summary['m2']['user_axis_allocation'] == (
    'b_N=q_C*pi_N, b_V=q_C*pi_V; b_N+b_V=q_C'
)
assert summary['m2']['learned_global_axis_weight'] is False
assert summary['m2']['raw_repeatshare_input'] is False
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['sample_weighting'] is False
assert summary['fixed']['one_training_loop_and_optimizer'] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = run_fixed_composition_screen(cfg)


In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
reading = dict(result_df.attrs['screening_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}

print('절대지표:')
display(display_df.sort_values('model_id'))

core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('M1@64 대비 핵심 변화:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', reading)
print('결과 파일:', paths)
